# Debt browser
Every instrument in the latest `cache/debt/debt_*.parquet` snapshot (`pricing/shared/debt_snapshot.py run`), filterable by CDS ticker, name text, currency, asset class, rank and size.
Needs the **system Python** kernel (ipywidgets / itables are not in the project .venv). Reads the file only; no Bloomberg.

In [4]:
import pathlib
import pandas as pd
import ipywidgets as w
from IPython.display import display, clear_output

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'cache' / 'debt').exists())
FILE = sorted((ROOT / 'cache' / 'debt').glob('debt_*.parquet'))[-1]          # newest snapshot
df = pd.read_parquet(FILE).sort_values(['market_cds_ticker', 'maturity'], na_position='last').reset_index(drop=True)
df['notional_usd_mm'] = (df.notional_outstanding / 1e6).round(1)
df['notional_local_mm'] = (df.notional_local / 1e6).round(1)
COLS = ['market_cds_ticker', 'security_name', 'asset_class', 'payment_rank', 'currency', 'issue_date', 'maturity', 'notional_usd_mm', 'notional_local_mm', 'coupon', 'security_id', 'equity_ticker']
print(f'{FILE.name}: {len(df):,} instruments, {df.market_cds_ticker.nunique()} names, pulled {df.pulled_at.max():%Y-%m-%d %H:%M}')

debt_2026-11-04.parquet: 13,129 instruments, 511 names, pulled 2026-11-04 10:41


In [5]:
ticker = w.SelectMultiple(options=sorted(df.market_cds_ticker.unique()), rows=8, description='Ticker', layout=w.Layout(width='260px'))
ticker_txt = w.Text(description='Ticker starts', placeholder='e.g. MP, HL', layout=w.Layout(width='260px'))
name_txt = w.Text(description='Name has', placeholder='e.g. TL B, 5 10/15', layout=w.Layout(width='260px'))
ccy = w.SelectMultiple(options=sorted(df.currency.dropna().unique()), rows=5, description='Currency', layout=w.Layout(width='160px'))
cls = w.SelectMultiple(options=sorted(df.asset_class.dropna().unique()), rows=3, description='Class', layout=w.Layout(width='200px'))
rank = w.SelectMultiple(options=sorted(df.payment_rank.dropna().unique()), rows=5, description='Rank', layout=w.Layout(width='260px'))
min_usd = w.FloatText(value=0, description='Min $mm', layout=w.Layout(width='160px'))
hide_zero = w.Checkbox(value=False, description='Hide undrawn / zero amount')
hide_matured = w.Checkbox(value=True, description='Hide matured (perpetuals kept)')
out = w.Output()

def refresh(*_):
    d = df
    if ticker.value:
        d = d[d.market_cds_ticker.isin(ticker.value)]
    if ticker_txt.value:
        d = d[d.market_cds_ticker.str.upper().str.startswith(ticker_txt.value.strip().upper())]
    if name_txt.value:
        d = d[d.security_name.str.contains(name_txt.value.strip(), case=False, regex=False, na=False)]
    if ccy.value:
        d = d[d.currency.isin(ccy.value)]
    if cls.value:
        d = d[d.asset_class.isin(cls.value)]
    if rank.value:
        d = d[d.payment_rank.isin(rank.value)]
    if min_usd.value:
        d = d[d.notional_usd_mm >= min_usd.value]
    if hide_matured.value:
        d = d[d.maturity.isna() | (d.maturity >= pd.Timestamp.today().normalize())]
    if hide_zero.value:
        d = d[d.notional_usd_mm > 0]
    with out:
        clear_output(wait=True)
        print(f'{len(d):,} instruments, {d.market_cds_ticker.nunique()} names, USD {d.notional_usd_mm.sum():,.0f} mm')
        with pd.option_context('display.max_rows', 500, 'display.max_columns', 30, 'display.width', 250, 'display.max_colwidth', 40):
            display(d[COLS].reset_index(drop=True).head(500))
        if len(d) > 500:
            print(f'(showing the first 500 of {len(d):,}: narrow the filters)')

for c in (ticker, ticker_txt, name_txt, ccy, cls, rank, min_usd, hide_zero, hide_matured):
    c.observe(refresh, names='value')
display(w.VBox([w.HBox([w.VBox([ticker_txt, ticker]), w.VBox([name_txt, min_usd, hide_matured, hide_zero]), ccy, cls, rank]), out]))
refresh()

Full table (sortable, searchable in the page) via itables, if you prefer a search box over the widgets:

In [6]:
from itables import show
show(df[df.maturity.isna() | (df.maturity >= pd.Timestamp.today().normalize())][COLS], maxBytes=0, lengthMenu=[25, 100, 500], classes='display compact')